In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor


sales = pd.read_csv(r"D:\foresight\data\raw\sales_daily.csv")

sales["date"] = pd.to_datetime(sales["date"])

print("Rows:", len(sales))
print("SKUs:", sales["sku_id"].nunique())

Rows: 548000
SKUs: 500


In [3]:
weekly_sales_all = (
    sales
    .groupby(
        ["sku_id", pd.Grouper(key="date", freq="W")]
    )["units_sold"]
    .sum()
    .reset_index()
    .sort_values(["sku_id", "date"])
)

print("Weekly rows:", len(weekly_sales_all))
print("SKUs:", weekly_sales_all["sku_id"].nunique())

Weekly rows: 79000
SKUs: 500


In [4]:
weekly_sales_all["lag_1"] = (
    weekly_sales_all
    .groupby("sku_id")["units_sold"]
    .shift(1)
)

weekly_sales_all["lag_2"] = (
    weekly_sales_all
    .groupby("sku_id")["units_sold"]
    .shift(2)
)

weekly_sales_all["lag_4"] = (
    weekly_sales_all
    .groupby("sku_id")["units_sold"]
    .shift(4)
)

weekly_sales_all["rolling_4"] = (
    weekly_sales_all
    .groupby("sku_id")["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(4).mean()
    )
)

weekly_sales_all["seasonal_naive"] = (
    weekly_sales_all
    .groupby("sku_id")["units_sold"]
    .shift(52)
)

In [5]:
model_data = weekly_sales_all.dropna(
    subset=[
        "lag_1",
        "lag_2",
        "lag_4",
        "rolling_4",
        "seasonal_naive"
    ]
).copy()

print("Rows:", len(model_data))
print("SKUs:", model_data["sku_id"].nunique())

Rows: 53000
SKUs: 500


In [6]:
features = [
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_4"
]

final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    model_data[features],
    model_data["units_sold"]
)

print("Final model trained.")

Final model trained.


In [7]:
latest_history = (
    weekly_sales_all
    .sort_values(["sku_id", "date"])
    .groupby("sku_id")
    .tail(4)
)

history = (
    latest_history
    .groupby("sku_id")["units_sold"]
    .apply(list)
    .to_dict()
)

forecast_table = pd.DataFrame({
    "sku_id": sorted(history.keys())
})

for week in range(1, 5):

    predictions = []

    for sku in forecast_table["sku_id"]:

        values = history[sku]

        model_input = pd.DataFrame([{
            "lag_1": values[-1],
            "lag_2": values[-2],
            "lag_4": values[-4],
            "rolling_4": np.mean(values[-4:])
        }])

        prediction = final_model.predict(model_input)[0]
        prediction = max(0, prediction)

        predictions.append(prediction)

    forecast_table[f"forecast_week_{week}"] = predictions

    for i, sku in enumerate(forecast_table["sku_id"]):
        history[sku].append(predictions[i])
        history[sku] = history[sku][-4:]

print(forecast_table.head())
print("Forecasted SKUs:", len(forecast_table))

KeyboardInterrupt: 

In [ ]:
inventory = pd.read_excel(r"D:\foresight\data\raw\inventory_snapshots.xlsx")

inventory["date"] = pd.to_datetime(inventory["date"])

latest_inventory = (
    inventory
    .sort_values("date")
    .groupby("sku_id")
    .tail(1)
    .copy()
)

print("Latest inventory SKUs:",
      latest_inventory["sku_id"].nunique())

Latest inventory SKUs: 500


In [ ]:
decision_data = latest_inventory.merge(
    forecast_table,
    on="sku_id",
    how="left"
)

print("Decision rows:", len(decision_data))

Decision rows: 500


In [ ]:
decision_data["forecast_4_weeks"] = (
    decision_data["forecast_week_1"]
    + decision_data["forecast_week_2"]
    + decision_data["forecast_week_3"]
    + decision_data["forecast_week_4"]
)

decision_data["avg_weekly_forecast"] = (
    decision_data["forecast_4_weeks"] / 4
)

In [ ]:
decision_data["lead_time_weeks"] = (
    decision_data["lead_time_days"] / 7
)

decision_data["lead_time_demand"] = (
    decision_data["avg_weekly_forecast"]
    * decision_data["lead_time_weeks"]
)

decision_data["available_inventory"] = (
    decision_data["on_hand_units"]
    + decision_data["on_order_units"]
)

decision_data["stockout_gap"] = (
    decision_data["lead_time_demand"]
    - decision_data["available_inventory"]
)

decision_data["stockout_risk"] = np.where(
    decision_data["stockout_gap"] > 0,
    "HIGH",
    "LOW"
)

In [ ]:
decision_data["overstock_threshold"] = (
    decision_data["forecast_4_weeks"] * 1.5
)

decision_data["overstock_risk"] = np.where(
    decision_data["on_hand_units"]
    > decision_data["overstock_threshold"],
    "HIGH",
    "LOW"
)

In [ ]:
recent_8 = (
    weekly_sales_all
    .sort_values(["sku_id", "date"])
    .groupby("sku_id")
    .tail(8)
)

volatility = (
    recent_8
    .groupby("sku_id")["units_sold"]
    .agg(
        demand_mean="mean",
        demand_std="std"
    )
    .reset_index()
)

volatility["cv"] = (
    volatility["demand_std"]
    / volatility["demand_mean"].replace(0, np.nan)
)

decision_data = decision_data.merge(
    volatility[["sku_id", "cv"]],
    on="sku_id",
    how="left"
)

In [ ]:
sku_master = pd.read_excel(
    "../data/raw/sku_master.xlsx"
)

decision_data = decision_data.merge(
    sku_master[
        [
            "sku_id",
            "category",
            "subcategory",
            "unit_cost",
            "list_price"
        ]
    ],
    on="sku_id",
    how="left"
)

print(decision_data.shape)
print(decision_data.columns.tolist())

(500, 26)
['date', 'sku_id', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point', 'safety_stock', 'forecast_week_1', 'forecast_week_2', 'forecast_week_3', 'forecast_week_4', 'forecast_4_weeks', 'avg_weekly_forecast', 'lead_time_weeks', 'lead_time_demand', 'available_inventory', 'stockout_gap', 'stockout_risk', 'overstock_threshold', 'overstock_risk', 'cv', 'action', 'category', 'subcategory', 'unit_cost', 'list_price']


In [ ]:
def get_action(row):

    if row["stockout_risk"] == "HIGH":
        return "REORDER NOW"

    if row["overstock_risk"] == "HIGH":
        return "MARKDOWN/CLEAR"

    if pd.notna(row["cv"]) and row["cv"] > 0.75:
        return "WATCH/VOLATILE"

    return "HEALTHY"


decision_data["action"] = decision_data.apply(
    get_action,
    axis=1
)

print(
    decision_data["action"].value_counts()
)

action
HEALTHY        401
REORDER NOW     99
Name: count, dtype: int64


In [ ]:
decision_data.to_csv(r"D:\foresight\data\foresight_decisions.csv",index=False)

forecast_table.to_csv(r"D:\foresight\data\foresight_forecasts.csv",index=False)

print("Dashboard data saved.")

Dashboard data saved.


In [ ]:
decision_data.shape
decision_data.head()

,date,sku_id,on_hand_units,on_order_units,lead_time_days,reorder_point,safety_stock,forecast_week_1,forecast_week_2,forecast_week_3,...,stockout_gap,stockout_risk,overstock_threshold,overstock_risk,cv,action,category,subcategory,unit_cost,list_price
0,2025-12-31,SKU335,118,102,7,51,22,124.365,147.380,142.060,...,-79.901250,LOW,840.5925,LOW,0.250055,HEALTHY,Dining,Tray,6757,10811
1,2025-12-31,SKU332,448,128,7,78,52,154.290,119.715,149.935,...,-435.827500,LOW,841.0350,LOW,0.255940,HEALTHY,Dining,Cutlery,6916,11065
2,2025-12-31,SKU333,498,75,10,94,46,130.855,151.000,135.935,...,-371.517857,LOW,846.2250,LOW,0.337366,HEALTHY,Dining,Bottle,6485,10376
3,2025-12-31,SKU334,39,63,10,37,19,150.240,138.205,143.995,...,100.882143,HIGH,852.1050,LOW,0.252368,REORDER NOW,Dining,Bottle,6789,10862
4,2025-12-31,SKU336,367,114,10,31,23,134.865,143.880,146.340,...,-277.689286,LOW,853.9050,LOW,0.328672,HEALTHY,Dining,Bottle,1594,2550


In [ ]:
import pandas as pd
import numpy as np

decision_data = pd.read_csv(r"D:\foresight\data\foresight_decisions.csv")

print("Rows:", len(decision_data))
print("SKUs:", decision_data["sku_id"].nunique())

Rows: 500
SKUs: 500


In [9]:
print("=== Stockout gap ===")
print(
    decision_data["stockout_gap"].describe()
)

print("\n=== Overstock ratio ===")
decision_data["stock_ratio"] = (
    decision_data["on_hand_units"]
    / decision_data["forecast_4_weeks"].replace(0, np.nan)
)

print(
    decision_data["stock_ratio"].describe()
)

print("\n=== Volatility (CV) ===")
print(
    decision_data["cv"].describe()
)

=== Stockout gap ===
count    500.000000
mean    -163.346379
std      172.452179
min     -543.954821
25%     -291.959509
50%     -160.444107
75%      -23.013616
max      226.470000
Name: stockout_gap, dtype: float64

=== Overstock ratio ===
count    500.000000
mean       0.438210
std        0.262135
min        0.018469
25%        0.195661
50%        0.429444
75%        0.675470
max        0.917879
Name: stock_ratio, dtype: float64

=== Volatility (CV) ===
count    500.000000
mean       0.273679
std        0.046998
min        0.140634
25%        0.240524
50%        0.273890
75%        0.302879
max        0.405556
Name: cv, dtype: float64


In [10]:
volatility_threshold = decision_data["cv"].quantile(0.90)

print("90th percentile CV:", volatility_threshold)

90th percentile CV: 0.33448477684234545


In [11]:
def get_action(row):

    if row["stockout_risk"] == "HIGH":
        return "REORDER NOW"

    if row["overstock_risk"] == "HIGH":
        return "MARKDOWN/CLEAR"

    if pd.notna(row["cv"]) and row["cv"] >= volatility_threshold:
        return "WATCH/VOLATILE"

    return "HEALTHY"


decision_data["action"] = decision_data.apply(
    get_action,
    axis=1
)

print(
    decision_data["action"].value_counts()
)

action
HEALTHY           362
REORDER NOW        99
WATCH/VOLATILE     39
Name: count, dtype: int64


In [ ]:
decision_data.to_csv(r"D:\foresight\data\foresight_decisions.csv",index=False)

In [14]:

print(
    "Missing list prices:",
    decision_data["list_price"].isna().sum()
)

Missing list prices: 0


In [16]:
decision_data["sales_at_risk"] = (
    decision_data["stockout_gap"]
    .clip(lower=0)
    * decision_data["list_price"]
)
print(
    "Total sales value at risk:",
    decision_data["sales_at_risk"].sum()
)

Total sales value at risk: 50239152.89678572


In [ ]:
decision_data.to_csv(r"D:\foresight\data\foresight_decisions.csv",index=False)